In [18]:
import os
import glob
import pandas as pd
from datetime import date, datetime
from dateutil.relativedelta import relativedelta

# Mendapatkan path direktori saat ini
current_directory = os.getcwd()

# Mencari semua file CSV di direktori yang sama dengan file .ipynb
csv_files = glob.glob(os.path.join(current_directory, "*_MT_SkuDescListing_*.csv"))

# Membaca semua file CSV dan skip 2 baris pertama
skudesc = []
for file in csv_files:
    df = pd.read_csv(file, skiprows=2)
    skudesc.append(df)

# Menggabungkan semua DataFrame menjadi satu
skudesc_combined = pd.concat(skudesc, ignore_index=True)

# Menghapus simbol '="' dan '"' di semua kolom string
skudesc_combined = skudesc_combined.apply(
    lambda x: x.str.replace('="', '').str.replace('"', '') if x.dtype == 'object' else x
)

# Mengubah kolom 'Short SKU' menjadi 8 karakter (leading zero)
skudesc_combined['Short SKU'] = skudesc_combined['Short SKU'].apply(lambda x: str(x).zfill(8))

# Menghapus kolom 'Unnamed'
skudesc_combined = skudesc_combined.loc[:, ~skudesc_combined.columns.str.contains('^Unnamed')]

# Ubah format tanggal menjadi YYYYMM
for col in ['Creation Date', 'Activated Date']:
    skudesc_combined[col] = pd.to_datetime(skudesc_combined[col], errors='coerce').dt.strftime('%Y%m')

C:\Users\alfa.pratama\AppData\Local\Temp\ipykernel_2008\1243356530.py:16: DtypeWarning: Columns (0: Tkt Type, 1: Item Grade, 2: Item Packing, 3: Launching Date, 4: Item Unit Size UOM, 5: Expiry Guideline Takeout) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, skiprows=2)
C:\Users\alfa.pratama\AppData\Local\Temp\ipykernel_2008\1243356530.py:16: DtypeWarning: Columns (0: Tkt Type, 1: Item Packing, 2: Launching Date, 3: Item Unit Size UOM, 4: Expiry Guideline Takeout, 5: Discontinue Date, 6: Hold Order Start Date, 7: Hold Order End Date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, skiprows=2)


In [19]:
# Pilih hanya kolom yang dibutuhkan
columns_to_keep = ['Line', 'Division', 'Department', 'Short SKU', 'Description', 'Creation Date', 'Activated Date', 'Retail W GST', 'Returnable?']
skudesc_combined = skudesc_combined[columns_to_keep]

In [ ]:
# # Generate bulan & tahun sekarang (format YYYYMM)
# today_month = (date.today() - relativedelta(months=1)).strftime('%Y%m')
# print("Periode sekarang:", today_month)

# Hanya pakai bila ingin menggunakan periode statis
today_month = ("202603")

# 🔹 Tambahkan kolom baru berisi periode sekarang
skudesc_combined['Period'] = today_month

# Tampilkan hasil
print(skudesc_combined.head())

   Line Division Department    Short SKU                      Description  \
0  ="9"    ="91"    ="9047"  ="00228855"       DELIVERY FEE - ORDER BY WA   
1  ="9"    ="91"    ="9047"  ="00234108"            PERSONAL SHOPEER FEE    
2  ="9"    ="91"    ="9047"  ="00238106"  DELIVERY FEE WVAT - ORDER BY WA   
3  ="9"    ="91"    ="9047"  ="00239042"                     WRAP REGULER   
4  ="9"    ="91"    ="9047"  ="00239059"                       WRAP EXTRA   

  Creation Date Activated Date  Retail W GST Returnable?  Period  
0        202003         202003           1.0           N  202603  
1        202105         202105       20000.0           N  202603  
2        202108         202108           1.0           N  202603  
3        202112         202112       15000.0           N  202603  
4        202112         202112       30000.0           N  202603  


: 

In [ ]:
import os
import pandas as pd
import glob
import tempfile
import shutil

# --- Konfigurasi ---
pd.set_option('display.max_columns', None)

current_dir = os.getcwd()
csv_files = glob.glob(os.path.join(current_dir, "*Daily Sales*.csv"))

# Buat folder sementara untuk simpan hasil preprocess per file
temp_dir = tempfile.mkdtemp()
print(f"📁 Temp directory: {temp_dir}")

processed_files = []

# --- Tahap 1: Proses tiap file satu per satu & simpan sementara ---
for i, file in enumerate(csv_files):
    print(f"🔄 Memproses file {i+1}/{len(csv_files)}: {os.path.basename(file)}")
    
    try:
        # Baca file (skip baris sesuai kebutuhan)
        df = pd.read_csv(file, skiprows=[0,1,2,3,4,5,6,8])
        
        # Validasi: pastikan kolom 'Date' dan 'Item' ada
        if 'Date' not in df.columns or 'Item' not in df.columns:
            print(f"  ⚠️ Kolom 'Date' atau 'Item' tidak ditemukan di {file}. Lewati.")
            continue
        
        # Bersihkan kolom 'Date' dan 'Item'
        df['Date'] = df['Date'].astype(str).str.replace(r'["=]', '', regex=True)
        df = df.rename(columns={'Item': 'Short SKU'})
        df['Short SKU'] = df['Short SKU'].astype(str).str.replace(r'["=]', '', regex=True)
        df['Short SKU'] = df['Short SKU'].str.rjust(8, '0')
        
        # Filter kolom: Date, Short SKU, dan yang mulai dengan 10/51/70
        cols_to_keep = ['Date', 'Short SKU'] + [
            col for col in df.columns 
            if str(col).startswith(('10', '51', '70'))
        ]
        df = df[cols_to_keep]
        
        # Gabungkan kolom offline (10xx) dan online (51xx)
        raisa = [(1001, 5101), (1002, 5102), (1003, 5103), (1004, 5104), (1006, 5106)]
        for offline, online in raisa:
            off_str, on_str = str(offline), str(online)
            if off_str in df.columns and on_str in df.columns:
                df[off_str] = df[off_str].fillna(0) + df[on_str].fillna(0)
                df = df.drop(columns=[on_str])
        
        # Simpan ke file sementara (format Parquet lebih efisien)
        temp_file = os.path.join(temp_dir, f"processed_{i:04d}.parquet")
        df.to_parquet(temp_file, index=False)
        processed_files.append(temp_file)
        
    except Exception as e:
        print(f"  ❌ Error memproses {file}: {e}")

print(f"\n✅ Selesai memproses {len(processed_files)} file.")

# --- Tahap 2: Gabungkan semua hasil sementara ---
print("📂 Menggabungkan semua data sementara...")

# Baca semua file sementara dan gabung
df_list = [pd.read_parquet(f) for f in processed_files]
df_combined = pd.concat(df_list, ignore_index=True)

# Hapus folder sementara
shutil.rmtree(temp_dir)
print("🗑️  File sementara dihapus.")

# --- Tahap 3: Transformasi akhir (melt + pivot) ---
print("⚙️  Melakukan melt dan pivot...")

# Melt ke format long
df_long = df_combined.melt(
    id_vars=['Date', 'Short SKU'],
    var_name='Store_Code',
    value_name='Sales'
)

# Hapus baris dengan Sales NaN atau 0 (opsional, sesuaikan kebutuhan)
df_long = df_long.dropna(subset=['Sales'])
df_long['Sales'] = pd.to_numeric(df_long['Sales'], errors='coerce').fillna(0)

# Pivot ke format lebar (hasil akhir)
df_pivot = df_long.pivot_table(
    index=['Store_Code', 'Short SKU'],
    columns='Date',
    values='Sales',
    aggfunc='sum'
).reset_index()

# Reset nama kolom agar rapi
df_pivot.columns.name = None

print("✅ Proses selesai!")
print(f"📊 Hasil akhir: {df_pivot.shape[0]} baris, {df_pivot.shape[1]} kolom")

📁 Temp directory: C:\Users\ALFA~1.PRA\AppData\Local\Temp\tmpnnv4ma1r
🔄 Memproses file 1/3: 2026001. Daily Sales Jan 2026.csv
🔄 Memproses file 2/3: 2026004. Daily Sales Apr 2026.csv
🔄 Memproses file 3/3: 2026005. Daily Sales May 2026.csv

✅ Selesai memproses 3 file.
📂 Menggabungkan semua data sementara...
🗑️  File sementara dihapus.
⚙️  Melakukan melt dan pivot...


In [ ]:
df_merge = pd.merge(df_pivot, skudesc_combined, on='Short SKU', how='left')

In [ ]:
import pandas as pd
import numpy as np

def detect_date_columns(df):
    date_cols = []
    for col in df.columns:
        try:
            pd.to_datetime(col)
            date_cols.append(col)
        except:
            pass
    return date_cols

# Salin df_merge ke DataFrame baru
df_activedate = df_merge.copy()

# Deteksi kolom tanggal (nama kolom berupa tanggal)
sales_cols = detect_date_columns(df_activedate)
print("📅 Kolom tanggal terdeteksi:", sales_cols)

# --- ✅ OPTIMISASI: HINDARI apply(axis=1) ---
# Buat kolom Effective_Date: Activated Date jika ada, else Creation Date
df_activedate['Effective_Date'] = df_activedate['Activated Date'].fillna(df_activedate['Creation Date'])

# Pastikan kolom Period dan Effective_Date bertipe string (karena format YYYYMM)
# Tapi perbandingan string "YYYYMM" tetap valid secara kronologis!

for col in sales_cols:
    if col not in df_activedate.columns:
        continue
        
    # Kondisi 1: NaN dan Period == Activated Date → biarkan NaN
    cond1 = df_activedate[col].isna() & (df_activedate['Period'] == df_activedate['Activated Date'])
    
    # Kondisi 2: NaN dan Period > Effective_Date → isi 0
    cond2 = df_activedate[col].isna() & (df_activedate['Period'] > df_activedate['Effective_Date'])
    
    # Update kolom secara vektoris
    df_activedate[col] = np.where(
        cond1,
        np.nan,
        np.where(cond2, 0, df_activedate[col])
    )

# Opsional: hapus kolom bantuan
df_activedate.drop(columns=['Effective_Date'], inplace=True)

print("✅ Logika active date selesai (lebih cepat!)")

In [ ]:
from scipy.stats import trim_mean
import numpy as np

# Pastikan kolom sales_cols numerik
df_activedate[sales_cols] = df_activedate[sales_cols].apply(pd.to_numeric, errors='coerce')

# Fungsi helper
def calc_trimmed_mean(row, cut=0.2):
    clean = row[~np.isnan(row)]
    return trim_mean(clean, cut) if len(clean) > 0 else np.nan

# Hitung
sales_array = df_activedate[sales_cols].values
df_activedate['ADS'] = np.apply_along_axis(
    calc_trimmed_mean, axis=1, arr=sales_array, cut=0.2
)

print("✅ Kolom 'ADS' berhasil ditambahkan.")

In [ ]:
import pandas as pd
import numpy as np

# Salin df_activedate agar tidak mengubah aslinya (opsional tapi disarankan)
df_final = df_activedate.copy()

# 1. Buat kolom CODE = STORE + SKU (pastikan keduanya string)
df_final['Store_Code'] = df_final['Store_Code'].astype(str)
df_final['Short SKU'] = df_final['Short SKU'].astype(str)
df_final['CODE'] = df_final['Store_Code'] + df_final['Short SKU']

# 2. Buat kolom ROUND = round(ADS)
# Pastikan ADS numerik dan handle NaN
df_final['ROUND'] = df_final['ADS'].round().astype('Int64')  # Int64 untuk pertahankan NaN sebagai <NA>

# 3. Rename kolom
df_final = df_final.rename(columns={
    'Store_Code': 'STORE',
    'Short SKU': 'SKU'
})

# Filter ADS yang kosong
filtered_ads = df_final[~df_final['ADS'].isin([0])]

# 5. Urutkan kolom sesuai permintaan
required_columns = ["CODE", "STORE", "SKU", "ROUND", "ADS"]
filtered_ads = filtered_ads[required_columns]

# 6. Ekspor ke Excel
output_file = "ADS OLD JUN 26 TRIM 90.csv"
filtered_ads.to_csv(output_file, index=False)

print(f"✅ Ekspor berhasil! File disimpan sebagai: {output_file}")
print(f"📊 Shape: {filtered_ads.shape}")
print(filtered_ads.head())